# Clustering deep-sea coral communities by taxon

We group survey events by the coral families recorded together. The resulting labels describe patterns in the observations and provide an input for Issue #11.

**Setup.** Install the repository requirements and `scikit-learn` in the notebook kernel. Run `ingest.ipynb` and `clean.ipynb` first, or use the team's matching Parquet snapshot. This notebook also needs a NOAA metadata extract keyed by catalog number. It downloads and caches the extract on first run. Preserve both inputs together to reproduce the analysis. The notebook runs from the repository root or the `notebooks` directory.

The verified run used Python 3.12, pandas 3.1.1, scikit-learn 1.9.1, and SciPy 1.17.1.

In [1]:
from pathlib import Path
from tempfile import NamedTemporaryFile

import pandas as pd
import requests
from sklearn.cluster import DBSCAN, KMeans
from sklearn.metrics import silhouette_score
from threadpoolctl import threadpool_limits

repo_dir = Path.cwd()
if repo_dir.name == "notebooks":
    repo_dir = repo_dir.parent

data_dir = repo_dir / "data"
cleaned_path = data_dir / "processed" / "se_coral_records_cleaned.parquet"
interim_path = data_dir / "interim" / "se_coral_records.parquet"
input_path = cleaned_path if cleaned_path.is_file() else interim_path

if not input_path.is_file():
    raise FileNotFoundError("Run ingest.ipynb or place the team Parquet snapshot under data/.")

records = pd.read_parquet(input_path)
print(f"Input: {input_path.relative_to(repo_dir)} ({len(records):,} records)")

if input_path == interim_path:
    observation_dates = pd.to_datetime(records["obs_date"], format="%Y-%m-%d", errors="coerce")
    records = records.assign(obs_date=observation_dates).dropna(subset=["obs_date"])
    print(f"Records with valid dates: {len(records):,}")

Input: data\processed\se_coral_records_cleaned.parquet (71,499 records)


## Identify coral records and sampling metadata

The source includes corals, sponges, and fish. We join NOAA's taxonomic category, dataset ID, data provider, and sampling equipment to the Parquet file using the unique NOAA catalog number. The metadata request uses the same regional, year, and deep-water filters as `ingest.ipynb`. The join checks that every input record has a metadata match before selecting the coral categories. The [NOAA ERDDAP data dictionary](https://www.ncei.noaa.gov/erddap/tabledap/deep_sea_corals.html) documents the source fields.

In [2]:
metadata_path = data_dir / "raw" / "se_coral_taxon_metadata.csv"
metadata_columns = {
    "CatalogNumber": "catalog_num",
    "DatasetID": "dataset_id",
    "DatabaseVersion": "database_version",
    "VernacularNameCategory": "taxon_category",
    "DataProvider": "data_provider",
    "SamplingEquipment": "sampling_equipment",
}
if not metadata_path.is_file():
    query_columns = ",".join(metadata_columns)
    ecosystems = "%28Southeast%20U.S.%20Continental%20Shelf%7CGulf%20of%20Mexico%7CCaribbean%20Sea%29"
    metadata_url = (
        "https://www.ncei.noaa.gov/erddap/tabledap/deep_sea_corals.csvp?"
        f"{query_columns}&ShallowFlag=0&ObservationYear%3E=2018&Country=%22USA%22"
        f"&LargeMarineEcosystem=~%22{ecosystems}%22"
    )
    metadata_path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = None
    try:
        with requests.get(metadata_url, stream=True, timeout=(20, 180)) as response:
            response.raise_for_status()
            with NamedTemporaryFile(dir=metadata_path.parent, delete=False) as temp_file:
                temporary_path = Path(temp_file.name)
                temp_file.writelines(response.iter_content(1 << 20))
        temporary_path.replace(metadata_path)
    finally:
        if temporary_path is not None:
            temporary_path.unlink(missing_ok=True)

metadata = pd.read_csv(
    metadata_path,
    usecols=list(metadata_columns),
    dtype={"CatalogNumber": "int64"},
    keep_default_na=False,
).rename(columns=metadata_columns)
matched_records = records.merge(metadata, on="catalog_num", how="left", validate="one_to_one", indicator=True)
unmatched_count = matched_records["_merge"].ne("both").sum()
if unmatched_count:
    raise ValueError(f"{unmatched_count:,} records lack catalog matches. Use matching NOAA snapshots.")
matched_records = matched_records.drop(columns="_merge")
versions = ", ".join(sorted(matched_records["database_version"].unique()))
print(f"NOAA database version in matched metadata: {versions}")

NOAA database version in matched metadata: 20260416-1


## Build the event-by-family matrix

We start with coral and sea pen categories from NOAA and require a known family and usable event identifiers. The event key combines survey ID and event ID. In this snapshot, the 35 modeled keys shared by multiple source datasets had overlapping dates and source median locations within 0.005 degrees of one another on each axis. We pool those records by event and retain their source IDs for provenance. Each family is counted once per event. A zero means the family was not recorded in the retained observations. Survey effort and taxonomic resolution can still affect the pattern.

AphiaID identifies a taxon and is kept with the observation. Counts, density, sample area, location, date, record type, location accuracy, provider, and equipment give context for interpreting the groups. Family presence is the clustering input.

In [3]:
coral_categories = {
    "alcyonacean (unspecified)", "black coral", "gorgonian coral", "lace coral",
    "scleralcyonacea (unspecified)", "sea pen", "soft coral", "stony coral (branching)",
    "stony coral (cup coral)", "stony coral (unspecified)",
}
event_keys = ["survey_id", "event_id"]
missing_identifiers = {"", "-1", "na", "n/a", "none", "unknown", "null", "nan", "not provided"}
corals = matched_records.loc[matched_records["taxon_category"].isin(coral_categories)].copy()
coral_count = len(corals)

for column in [*event_keys, "family"]:
    values = corals[column].astype("string").str.strip()
    corals[column] = values.mask(values.str.casefold().isin(missing_identifiers))

known_family_count = corals["family"].notna().sum()
corals = corals.dropna(subset=[*event_keys, "family"])
if corals.empty:
    raise ValueError("No coral records have both a usable family and an event key.")

event_families = corals.groupby([*event_keys, "family"]).size().unstack(fill_value=0)
family_presence = event_families.gt(0).astype("uint8").sort_index()
family_richness = family_presence.sum(axis=1).rename("recorded_families")

print(f"Coral records: {coral_count:,}")
print(f"Coral records with a known family: {known_family_count:,}")
print(f"Coral records with both family and event key: {len(corals):,}")
print(f"Modeled events: {len(family_presence):,}")
print(f"Family features: {family_presence.shape[1]:,}")
print(f"Events with only one recorded family: {family_richness.eq(1).mean():.1%}")
print(f'Survey/event keys shared across sources: {corals.groupby(event_keys)["dataset_id"].nunique().gt(1).sum():,}')

Coral records: 58,306
Coral records with a known family: 46,425
Coral records with both family and event key: 46,364
Modeled events: 275
Family features: 43
Events with only one recorded family: 12.0%
Survey/event keys shared across sources: 35


## Fit and compare K-Means models

K-Means is the baseline. Each feature uses the same 0–1 scale, so the cluster centroid for a family is the fraction of cluster events where that family was recorded. We compare cluster counts on the same data with one reproducible silhouette sample. Silhouette measures separation in the fitted feature space. Inertia describes within-cluster distance and tends to decrease as we add clusters. Cluster sizes help reveal whether a higher score comes with very small groups.

In [4]:
features = family_presence.to_numpy(dtype="float32")
max_clusters = min(8, len(features) - 1, len(family_presence.drop_duplicates()))
if max_clusters < 2:
    raise ValueError("At least three events and two distinct family profiles are needed.")

seed = 42
sample_size = min(2_000, len(features))
metrics = []
models = {}

with threadpool_limits(limits=1):
    for k in range(2, max_clusters + 1):
        model = KMeans(n_clusters=k, n_init=10, random_state=seed).fit(features)
        cluster_sizes = pd.Series(model.labels_).value_counts()
        score = silhouette_score(features, model.labels_, sample_size=sample_size, random_state=seed)
        models[k] = model
        metrics.append({
            "model": "K-Means",
            "k": k,
            "clusters": len(cluster_sizes),
            "distance": "Euclidean",
            "noise_events": 0,
            "noise_share": 0.0,
            "silhouette": score,
            "inertia": model.inertia_,
            "smallest_cluster": int(cluster_sizes.min()),
            "largest_cluster": int(cluster_sizes.max()),
            "evaluated_events": sample_size,
        })

evaluation = pd.DataFrame(metrics)
selected_k = int(evaluation.loc[evaluation["silhouette"].idxmax(), "k"])
evaluation["selected"] = evaluation["k"].eq(selected_k)
labels = pd.Series(models[selected_k].labels_, index=family_presence.index, name="cluster")
print(f"Selected k = {selected_k} by highest silhouette on the common sample.")
display(evaluation.round({"silhouette": 4, "inertia": 2}))

Selected k = 2 by highest silhouette on the common sample.

,model,k,clusters,distance,noise_events,noise_share,silhouette,inertia,smallest_cluster,largest_cluster,evaluated_events,selected
0,K-Means,2,2,Euclidean,0,0.0,0.2871,821.93,45,230,275,True
1,K-Means,3,3,Euclidean,0,0.0,0.1912,693.91,39,135,275,False
2,K-Means,4,4,Euclidean,0,0.0,0.1936,623.12,37,94,275,False
3,K-Means,5,5,Euclidean,0,0.0,0.2242,558.32,16,85,275,False
4,K-Means,6,6,Euclidean,0,0.0,0.1951,529.27,16,71,275,False
5,K-Means,7,7,Euclidean,0,0.0,0.2116,499.97,16,82,275,False
6,K-Means,8,8,Euclidean,0,0.0,0.1922,480.60,16,46,275,False


## Check density-based groups with DBSCAN

The K-Means baseline leaves overlap between its groups. DBSCAN tests whether there are multiple dense clusters when comparing recorded family sets with Jaccard distance. Two events are closer when they share a larger fraction of their combined recorded families. We vary the neighborhood radius and fix the minimum number of events at five.

We score DBSCAN on assigned events when there are at least two clusters, and report noise for the whole dataset. Its silhouette scores use a different distance and can omit a large share of events, so direct ranking against the K-Means silhouette would be misleading. For downstream inspection, we retain labels from the tested radius with the lowest noise among runs that yield two or more clusters. The primary community labels come from K-Means.

In [5]:
presence = features.astype(bool)
density_metrics = []
density_runs = {}

with threadpool_limits(limits=1):
    for eps in (0.2, 0.3, 0.4, 0.5, 0.6, 0.7):
        assignments = DBSCAN(eps=eps, min_samples=5, metric="jaccard", algorithm="brute").fit_predict(presence)
        assigned = assignments != -1
        noise_count = int((~assigned).sum())
        sizes = pd.Series(assignments[assigned]).value_counts()
        scored_events = min(sample_size, int(assigned.sum()))
        score = None
        if 2 <= len(sizes) < scored_events:
            score = silhouette_score(
                presence[assigned], assignments[assigned], metric="jaccard",
                sample_size=scored_events, random_state=seed,
            )
        density_runs[eps] = assignments
        density_metrics.append({
            "model": "DBSCAN",
            "eps": eps,
            "min_samples": 5,
            "clusters": len(sizes),
            "distance": "Jaccard",
            "silhouette": score,
            "smallest_cluster": int(sizes.min()) if len(sizes) else None,
            "largest_cluster": int(sizes.max()) if len(sizes) else None,
            "evaluated_events": scored_events if score is not None else 0,
            "noise_events": noise_count,
            "noise_share": noise_count / len(presence),
            "selected": False,
        })

density_evaluation = pd.DataFrame(density_metrics)
multi_cluster = density_evaluation.loc[density_evaluation["clusters"].ge(2)]
density_labels = pd.Series(pd.NA, index=family_presence.index, dtype="Int64", name="dbscan_cluster")

if not multi_cluster.empty:
    representative_row = multi_cluster["noise_events"].idxmin()
    representative_eps = density_evaluation.loc[representative_row, "eps"]
    density_evaluation.loc[representative_row, "selected"] = True
    density_labels = pd.Series(density_runs[representative_eps], index=family_presence.index, name="dbscan_cluster")
    print(f"Retained DBSCAN labels for eps={representative_eps}; -1 denotes noise.")

display(density_evaluation.round({"silhouette": 4, "noise_share": 3}))

Retained DBSCAN labels for eps=0.4; -1 denotes noise.


,model,eps,min_samples,clusters,distance,silhouette,smallest_cluster,largest_cluster,evaluated_events,noise_events,noise_share,selected
0,DBSCAN,0.2,5,6,Jaccard,0.5351,5,68,137,138,0.502,False
1,DBSCAN,0.3,5,6,Jaccard,0.3627,5,132,194,81,0.295,False
2,DBSCAN,0.4,5,4,Jaccard,0.3250,7,179,233,42,0.153,True
3,DBSCAN,0.5,5,1,Jaccard,NaN,256,256,0,19,0.069,False
4,DBSCAN,0.6,5,1,Jaccard,NaN,261,261,0,14,0.051,False
5,DBSCAN,0.7,5,1,Jaccard,NaN,269,269,0,6,0.022,False


## Summarize families and export the results

The family profile is the share of events in a cluster with a record of each family. For each event, the location is the median latitude and longitude across its retained records that have a valid coordinate pair. This point summarizes those observations. It can hide the spatial extent of a dive or haul. Events without valid coordinates keep their labels.

The labels can be joined to original observations using the event key. The exports include the input family-presence matrix so a downstream notebook can reproduce separation plots with the same rows and features.

In [6]:
family_profiles = family_presence.groupby(labels).mean().rename_axis(index="cluster", columns=None)
cluster_summary = pd.concat(
    [labels.value_counts().sort_index().rename("events"), family_richness.groupby(labels).mean().rename("mean_families")],
    axis=1,
)
cluster_summary["event_share"] = cluster_summary["events"] / len(labels)
top_families = family_profiles.apply(lambda row: ", ".join(row[row.gt(0)].nlargest(3).index), axis=1)
cluster_summary["most_recorded_families"] = top_families
display(cluster_summary.round(3))

has_location = corals["lat"].between(-90, 90) & corals["lon"].between(-180, 180)
locations = corals.loc[has_location].groupby(event_keys)[["lat", "lon"]].median()
def list_values(series):
    values = {str(value).strip() for value in series.dropna()} - {""}
    return " | ".join(sorted(values))


event_metadata = corals.groupby(event_keys).agg(
    coral_records=("catalog_num", "size"),
    observation_start=("obs_date", "min"),
    observation_end=("obs_date", "max"),
    dataset_ids=("dataset_id", list_values),
    data_providers=("data_provider", list_values),
    sampling_equipment=("sampling_equipment", list_values),
)
event_labels = (
    labels.to_frame().join(density_labels).join(family_richness).join(event_metadata).join(locations).reset_index()
)
if event_labels.duplicated(event_keys).any() or len(event_labels) != len(family_presence):
    raise ValueError("Event labels do not align with modeled rows.")

output_dir = data_dir / "processed"
output_dir.mkdir(parents=True, exist_ok=True)
event_labels.to_parquet(output_dir / "taxon_cluster_events.parquet", index=False)
family_presence.reset_index().to_parquet(output_dir / "taxon_cluster_family_presence.parquet", index=False)
model_metrics = pd.concat([evaluation, density_evaluation], ignore_index=True)
model_metrics.to_csv(output_dir / "taxon_cluster_metrics.csv", index=False)
family_profiles.to_csv(output_dir / "taxon_cluster_family_profiles.csv")
print(f"Exported {len(event_labels):,} event labels and {len(model_metrics):,} model evaluations.")
print(f"Events with coordinates: {event_labels[['lat', 'lon']].notna().all(axis=1).sum():,}")

,events,mean_families,event_share,most_recorded_families
cluster,,,,
0,230,5.209,0.836,"Antipathidae, Ellisellidae, Plexauridae"
1,45,9.533,0.164,"Keratoisididae, Chrysogorgiidae, Primnoidae"


Exported 275 event labels and 13 model evaluations.
Events with coordinates: 275


## Interpretation of this run

K-Means selected two clusters, with 230 and 45 events and a silhouette of 0.287 across all 275 events. The smaller group recorded more families on average (9.53 versus 5.21) and more often included Keratoisididae, Chrysogorgiidae, and Primnoidae. Antipathidae, Ellisellidae, and Plexauridae were the most frequently recorded families in the larger group. The difference in recorded family richness could reflect real differences in composition, survey effort, or taxonomic identification.

DBSCAN identified four groups at a Jaccard radius of 0.4, with 42 events labeled as noise (15.3%). At 0.5 and above it produced one cluster plus noise. That sensitivity and the K-Means silhouette suggest the groups are useful summaries of the observations, with substantial overlap or dependence on how similarity is defined. The labels can be profiled and mapped in Issue #11. Evaluating whether environmental measurements predict those labels requires a separate analysis.